# သင်ခန်းစာ 11 - အေဂျင့်-မှ-အေဂျင့် (A2A) ပရိုတိုကော


## တပ်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ပရိုတိုကော ဆိုတာ ဘာလဲ?

ဤ **Agent-to-Agent (A2A) ပရိုတိုကော** သည် AI ကိုယ်စားလှယ်များအား အချင်းချင်း ဆက်သွယ်နိုင်စေရန်၊ ရှာဖွေတွေ့ရှိနိုင်စေရန်နှင့် ပူးပေါင်းဆောင်ရွက်နိုင်စေရန် ခွင့်ပြုသော ဖွင့်လှစ် စံနှုန်းတစ်ခုဖြစ်ပြီး၊
တစ်ဦးနှင့်တစ်ဦး ရှာဖွေတွေ့ရှိ၍၊ ပူးပေါင်းဆောင်ရွက်နိုင်စေသည် — မတူညီသော framework များပေါ်တွင် တည်ဆောက်ထားခြင်း သို့မဟုတ် hosted
ဖြစ်သော အခြား ဝန်ဆောင်မှုများပေါ်တွင် ဟိုစ့်လုပ်ထားခြင်းဖြစ်ပါစေ။

Key concepts:

- **Discovery** – ကိုယ်စားလှယ်များသည် ၎င်းတို့၏ စွမ်းရည်များကို ဖော်ပြသည့် *Agent Card* ကို ထုတ်ဝေ၍၊ ၎င်းက
  အခြား ကိုယ်စားလှယ်များ (သို့) အစီအစဉ်ညွှန်ကြားသူများအတွက် တာဝန်တစ်ခုအတွက် ကိုက်ညီသည့် ကျွမ်းကျင်သူကို ရှာဖွေရန် လွယ်ကူစေသည်။
- **Message Passing** – ကိုယ်စားလှယ်များသည် ပုံစံသတ်မှတ်ထားသော စာတိုများကို ပုံမှန် ပရိုတိုကောတစ်ခုဖြင့် လဲလှယ်ကြပြီး၊ အဲ့ဒါကြောင့် တစ်ဦး၏ တောင်းဆိုချက်ကို
  အတွင်းလုပ်ဆောင်ချက် ကွဲပြားချက်များရှိခဲ့လည်း တစ်ဦးမှ နားလည်ကာ ပြည့်မြောက်အောင် နောက်တစ်ဦးက ဆောင်ရွက်နိုင်သည်။
- **Task Lifecycle** – A2A သည် *တင်သွင်းခဲ့သည်*, *အလုပ်လုပ်နေသည်*, *ပြီးစီးထားသည်*, နှင့်
  *ပျက်ကွက်ခဲ့သည်* ကဲ့သို့ အခြေအနေများကို သတ်မှတ်ပေးကာ ခန့်အပ်ထားသော တာဝန်တစ်ခု ဘယ်လို တိုးတက်လျက်ရှိသည်ကို အစီအစဉ်ညွှန်ကြားသူအား လုံးဝမြင်သာစေသည်။

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## အထူးပြု ခရီးသွား ကိုယ်စားလှယ်များ ဖန်တီးခြင်း


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## အလုပ်စဉ်မှတဆင့် အေဂျင့်များ ပူးပေါင်းဆောင်ရွက်ခြင်း

ကျွန်ုပ်တို့သည် A2A မက်ဆေ့ခ်် ပို့ဆောင်မှုနှင့် ကိုက်ညီသည့် အဆင့်လိုက် အလုပ်စဉ်အဖြစ် အေဂျင့် သုံးဦးကို ချိတ်ဆက်ထားသည်။ 

1. **CurrencyExchangeAgent** သည် အသုံးပြုသူ၏ တောင်းဆိုချက်ကို လက်ခံပြီး ငွေကြေး ညွှန်ကြားချက်များ ပေးဆောင်သည်။
2. **ActivityPlannerAgent** သည် တိုးထွင်ပြည့်စုံထားသော အချက်အလက်များကို လက်ခံပြီး လှုပ်ရှားမှု အကြံပြုချက်များ ထည့်သွင်းပေးသည်။
3. **TravelManagerAgent** သည် နှစ်ဖက်၏ အချက်အလက်များကို ပေါင်းစည်း၍ နောက်ဆုံး ခရီးသွား အကျဉ်းချုပ် တစ်ခု ပြုစုသည်။


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ထုတ်လုပ်မှုပတ်ဝန်းကျင်တွင် A2A ကို နားလည်ခြင်း

ထုတ်လုပ်မှု ပတ်ဝန်းကျင်တွင် A2A ပရိုတိုကေါ် သည် ဝန်ဆောင်မှုများအကြား အင်အားကြီးသော စိန်ဖြေမှုများကို ဖွင့်ပေးသည်။

| Capability | Description |
|---|---|
| **Framework များကြား ပူးပေါင်းဆက်သွယ်နိုင်ခြင်း** | framework တစ်ခုဖြင့် ဆောက်ထားသော agent တစ်ခုသည် အခြား A2A-ကိုက်ညီသော framework များဖြင့် ဆောက်ထားသော agent မည်သည်ကိုမဆို တာဝန်များ ပေးပို့နိုင်ပြီး အဖွဲ့အစည်းများအချင်းချင်း အပြည့်အဝ ပူးပေါင်း ဆက်လုပ်နိုင်စေသည်။ |
| **ဝန်ဆောင်မှုနယ်နိမိတ်များ** | Agents များသည် သီးခြား microservices, cloud ဒေသများ သို့မဟုတ် သီးခြား အဖွဲ့အစည်းများတွင် တည်ရှိနိုင်ပြီး ထိရောက်စွာ ပူးပေါင်းဆောင်ရွက်နိုင်သည်။ |
| **ဒိုင်နမစ် ရှာဖွေခြင်း** | Orchestrator သည် runtime တွင် Agent Card စာရင်းကို မေးမြန်း၍ သတ်မှတ်ထားသော အပိုင်းတာဝန်အတွက် အကောင်းဆုံး သက်ဆိုင်ကျွမ်းကျင်သူကို ရှာဖွေနိုင်သည်။ |
| **Streaming နှင့် push အသိပေးချက်များ** | A2A သည် real-time အဆင့်တိုးတက်မှုအသိပေးမှုများအတွက် Server-Sent Events (SSE) ကို ထောက်ပံ့ပြီး ကြာရှည် ဆောင်ရွက်ရသည့် လုပ်ငန်းများအတွက် push အသိပေးချက်များကိုလည်း ထောက်ပံ့သည်။ |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ သင်သင်ယူခဲ့တာတွေမှာ:

1. **A2A ပရိုတိုကေါ် ဆိုတာဘာလဲ** — အေဂျင့်များအကြား ရှာဖွေရေး, စာတိုက်ဆက်သွယ်ရေး,
   နှင့် တာဝန်စီမံခန့်ခွဲမှု။
2. **အထူးပြု အေဂျင့်များကို ဘယ်လို ဖန်တီးမလဲ** — ငွေလဲလှယ်ရေး အေဂျင့်, လှုပ်ရှားမှု စီစဉ်ရေး အေဂျင့်,
   နှင့် ခရီးသွား မန်နေဂျာကို ညှိနှိုင်းပေးသူ။
3. **အေဂျင့်များကို လုပ်ငန်းစဉ်ထဲသို့ ချိတ်ဆက်နည်း** — `WorkflowBuilder` ကို အသုံးပြု၍ အဆက်လိုက်
   အေဂျင့်များအကြား စာတိုက်ပို့ခြင်းကို မော်ဒယ်ဖော်ခြင်း။
4. **A2A က ထုတ်လုပ်ရေးပတ်ဝန်းကျင်မှာ ဘယ်လို လည်ပတ်သည်** — framework မျိုးစုံနှင့် service မျိုးစုံများအကြား ပူးပေါင်းဆောင်ရွက်နိုင်စေခြင်း
   dynamic discovery နှင့် streaming updates များဖြင့်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
တာဝန်ပယ်ချ?,?,?,?,ခန်းချက်:

ဤစာရွက်စာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု Co-op Translator (https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်စွာ ဘာသာပြန်ရန် ကြိုးစားပေမယ့် အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ပြီး၊ ထိုကဲ့သို့သော အမှားများအတွက် အာမခံမထားပါ။ မူလဘာသာဖြင့် ရေးသားထားသည့် မူရင်းစာတမ်းကို တရားဝင်နှင့် အာဏာပိုင် အရင်းမြစ်အဖြစ် စဉ်းစားရမည်ဖြစ်သည်။ အရေးပါသော အချက်အလက်များအတွက် လူသား ပရော်ဖက်ရှင်နယ် ဘာသာပြန်ခြင်းကို မွမ်းမံစဉ်းစားရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုမှုကြောင့် ဖြစ်ပေါ်နိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မှားယွင်းသော အဓိပ္ပာယ်ဖော်ပြချက်များအတွက် ကျွန်ုပ်တို့မှာ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
